# Stage1 风险层 MA200 风险开启信号：只读必要支持

使用前一原生交易日拆分连续 SOXX 收盘与200日均线，对照原0bp实物账户现金分量。先前预览已在 plan.json 披露。


In [ ]:
import hashlib,json,os,pathlib,shlex
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/stage1_ma200_risk_release_support_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
component=root/src['component_card'];older=root/src['previous_MA200_rule_card'];artifact=component/'qlib_recorder'/src['component_experiment_id']/src['component_recorder_id']/'artifacts/analysis/daily'
for p,h in [(component/'summary.json',src['component_summary_sha256']),(artifact,src['component_daily_sha256']),(root/src['provider_uri']/'qlib_store_manifest.json',src['provider_manifest_sha256']),(older/'summary.json',src['previous_MA200_rule_summary_sha256'])]:assert sha(p)==h,(p,sha(p),h)
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['provider_uri']),region='us',kernels=1)
raw=D.features(['SOXX'],['$close'],start_time='2022-01-03',end_time='2026-10-07',freq='day');assert raw.index.get_level_values('instrument').unique().tolist()==['SOXX'];price=raw['$close'].droplevel('instrument').astype(float);assert price.index.is_unique and price.index.is_monotonic_increasing and price.gt(0).all();ma=price.rolling(200,min_periods=200).mean()
with R.uri_context(str(component/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['component_experiment_id'],recorder_id=src['component_recorder_id']);assert rec.status=='FINISHED';daily=rec.load_object('analysis/daily').copy()
assert len(daily)==944 and pd.DatetimeIndex(daily.signal_date).is_monotonic_increasing and pd.DatetimeIndex(daily.trade_date).is_monotonic_increasing
signals=pd.DatetimeIndex(daily.signal_date);completed_close=price.reindex(signals);completed_ma=ma.reindex(signals);assert completed_close.notna().all() and completed_ma.notna().all() and (signals<pd.DatetimeIndex(daily.trade_date)).all()
daily['SOXX_prior_close']=completed_close.to_numpy(float);daily['SOXX_prior_MA200']=completed_ma.to_numpy(float);daily['risk_on']=daily.SOXX_prior_close.gt(daily.SOXX_prior_MA200)
static=pd.concat({'feature':daily[['SOXX_prior_close','SOXX_prior_MA200','risk_on']].astype(float),'meta':daily[['cash_opportunity_log','SOXL_selection_log','paired_log_excess']]},axis=1);static.index=pd.MultiIndex.from_arrays([daily.trade_date,['SOXL_SOXX_CASH']*len(daily)],names=['datetime','instrument']);dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'evaluation':('2023-01-04','2026-10-08')});assert len(dataset.prepare('evaluation',col_set=['feature','meta']))==944
def describe(g):
 neg=g.loc[g.cash_opportunity_log.lt(0)];pos=g.loc[g.cash_opportunity_log.gt(0)];on=g.loc[g.risk_on];off=g.loc[~g.risk_on]
 return {'sessions':len(g),'risk_on_days':len(on),'risk_off_days':len(off),'cash_opportunity_log':float(g.cash_opportunity_log.sum()),'cash_log_risk_on':float(on.cash_opportunity_log.sum()),'cash_log_risk_off':float(off.cash_opportunity_log.sum()),'negative_cash_days':len(neg),'negative_cash_risk_on':int(neg.risk_on.sum()),'negative_cash_risk_off':int((~neg.risk_on).sum()),'positive_cash_days':len(pos),'positive_cash_risk_on':int(pos.risk_on.sum()),'positive_cash_risk_off':int((~pos.risk_on).sum()),'SOXL_selection_log':float(g.SOXL_selection_log.sum())}
annual={str(y):describe(g) for y,g in daily.groupby(pd.DatetimeIndex(daily.trade_date).year)};quarter={str(q):describe(g) for q,g in daily.groupby(pd.DatetimeIndex(daily.trade_date).to_period('Q'))};rolling=[]
for i in range(len(daily)-125):
 g=daily.iloc[i:i+126];rolling.append({'first_trade':str(pd.Timestamp(g.trade_date.iloc[0]).date()),'last_trade':str(pd.Timestamp(g.trade_date.iloc[-1]).date()),**describe(g)})
assert len(annual)==4 and len(quarter)==16 and len(rolling)==819
result={'status':'fixed_MA200_risk_on_does_not_align_original_Stage1_broad_cash_loss','plan_sha256':sha(card/'plan.json'),'all':describe(daily),'annual':annual,'quarterly':quarter,'rolling126_count':len(rolling),'Qlib_dataset_rows':944,'first_signal':str(signals[0].date()),'last_signal':str(signals[-1].date()),'source_price_calendar_rows':len(price),'new_model_fits':0,'new_native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
state={'status':'Recorder_started','plan_sha256':result['plan_sha256'],'new_model_fits':0,'new_native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset);rec.save_objects(artifact_path='analysis',daily=daily,annual=annual,quarterly=quarter,rolling126=pd.DataFrame(rolling),result=result);rec.log_metrics(evaluation_sessions=944,negative_cash_risk_off_2025=annual['2025']['negative_cash_risk_off'],new_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
summary={'experiment':plan['experiment'],'decision':'closed_before_account_MA200_risk_on_cap_release_targets_2024_protection_misses_2025_cash_loss','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':evidence,'NAS_readback':{'sha256':remote,'verified':True},'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/annual','analysis/quarterly','analysis/rolling126','analysis/result']},'limitations':['The previously researched standalone MA200 trend family stays closed.','The previewed cash component groups are original-account development attribution, not causal strategy returns.','No Stage1 risk-on cap override, Tiered candidate account, 2/5bp or independent forward evaluation ran.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'all':result['all'],'2024Q3':quarter['2024Q3'],'2025':annual['2025'],'2026Q3':quarter['2026Q3'],'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False,indent=2))


## 中文结论

使用原 Stage1 冻结完整 **0bp、仍扣 Tiered 费用**账户的944个交易日，只在09:46前读取上一原生交易日 SOXX 的拆分连续 Qlib 收盘与其截至该日的200交易日均价。原账户物理现金机会与 SOXL相对选择分量来自已核对的Recorder；全年、16季度、819个连续126日窗口和所有日级真假状态保存在新Recorder。此前独立SOXL/cash的MA200趋势规则族已失败，本卡只检查把同一信息接到 Stage1 风险层是否有必要支持，**没有构造资金反事实**。

关键差区间与信号错位。2025原账户现金机会对数贡献 **-0.15801**，其中均线下方 **-0.16054**，均线上方 **+0.00253**；因此“均线上方放松风险上限”没有覆盖主要现金拖累。2024Q3原有现金保护 **+0.06444**，其中均线上方贡献 **+0.09477**，简单放松反而会选中有效保护区间。2026Q3 **64/64** 日均在均线上方，原账户 SOXL相对选择对数损失 **-0.14390**，提高风险也不能从该季度现金分量单独推出净超额。

决定 **`closed_before_account_MA200_risk_on_cap_release_targets_2024_protection_misses_2025_cash_loss`**。先前预览已在计划中披露，这些2023–2026分组是事后发展反馈；不改均线周期、方向、ETF或年份来救援，不追加 Stage1 0bp 资金账户。此必要支持失败不证明所有长趋势规则无效，也不证明完整账户的反馈必然亏损。

原生 Qlib DatasetH/Recorder `427867c6f2be4ff89f91373a28f376b1` 及 NAS SHA-256 `bae30d7d6c4eeb6c461b8d4cf7946a36809573571ff62a388aa6a0d44acfc82f` 已读回、五件工件限定冷恢复一致。0拟合、0账户、0bootstrap；SOXX稳定超额、2/5bp和独立前向证据仍未得到通过。
